# 製程改善統計檢定：完整示範

以內建的製程範例資料，走一遍八種檢定（對照 Excel 資料分析工具箱）。
每個檢定的**使用時機／前提假設**都在函數 docstring 裡，可用 `help()` 查閱。

In [ ]:
import process_validation as pv
from process_validation import datasets as ds

help(pv.f_test)  # 看看備註說明長什麼樣子

## 步驟 0：前提假設檢查（Excel 做不到）

t / F / ANOVA 都假設資料近似常態；等變異 t 另外要求變異數同質。先檢查：

In [ ]:
display(pv.check_normality({'改善前': ds.WELCH_BEFORE, '改善後': ds.WELCH_AFTER}))
display(pv.check_equal_variance({'改善前': ds.WELCH_BEFORE, '改善後': ds.WELCH_AFTER}))

## 1. F 檢定：改善後變異（穩定度）是否縮小？

In [ ]:
print(pv.f_test(ds.F_BEFORE, ds.F_AFTER, tail='one'))

## 2. 成對 t 檢定：同 10 台設備調機前後的節拍時間

In [ ]:
print(pv.t_test_paired(ds.PAIRED_BEFORE, ds.PAIRED_AFTER))

## 3. 等變異 t 檢定：新舊治具批次良率

（F 檢定不顯著 → 可用等變異版本）

In [ ]:
print(pv.t_test_equal_var(ds.EQVAR_OLD, ds.EQVAR_NEW))

## 4. 不等變異 t 檢定（Welch）：改善前後良率（變異明顯不同）

In [ ]:
print(pv.t_test_welch(ds.WELCH_BEFORE, ds.WELCH_AFTER))

## 5. 卡方獨立性檢定：改善前後 × 良/不良（不良率是否下降）

In [ ]:
print(pv.chi_square_independence(ds.CHI_IND_TABLE))

## 6. 卡方適合度檢定：缺陷類型分布是否符合歷史比例

In [ ]:
print(pv.chi_square_gof(ds.CHI_GOF_OBSERVED, ds.CHI_GOF_EXPECTED))

## 7. 單因子 ANOVA：三種退火溫度 vs 硬度

顯著後接 Tukey HSD 找出是哪幾組不同：

In [ ]:
r = pv.anova_oneway(ds.ANOVA1_GROUPS)
print(r)
if r.significant:
    display(pv.tukey_hsd(ds.ANOVA1_GROUPS))

## 8. 雙因子 ANOVA：溫度 × 壓力（重複試驗，含交互作用）

**判讀順序：先看交互作用**——顯著時直接比較「各組合平均」選最佳組合。

In [ ]:
print(pv.anova_twoway(ds.ANOVA2_REP))

### 無重複試驗版本：操作員 × 機台（每格 1 筆，無法估交互作用）

In [ ]:
print(pv.anova_twoway(ds.ANOVA2_NOREP))

## 輸出 Excel 報告

In [ ]:
result = pv.t_test_welch(ds.WELCH_BEFORE, ds.WELCH_AFTER)
result.to_excel('檢定報告.xlsx')
print('已輸出 檢定報告.xlsx（摘要 + 敘述統計各一張工作表）')